# 06 — Shadow Geometry False-Positive Filter

**DRISHTI** — Deterministic geometric check to suppress rock-cluster false positives

A rock cluster gives a strong sonar return but its shadow won't match what
geometry predicts for its apparent height, while a real raised object's shadow will.
This notebook demonstrates the trigonometric shadow verification.

**Run on:** Any environment (CPU only, pure geometry).

In [ ]:
!pip install -q numpy matplotlib opencv-python-headless

In [ ]:
import numpy as np
import cv2
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
print('Setup complete.')

## 1. Sonar Shadow Geometry

In side-scan sonar, objects cast acoustic shadows. The shadow length is
determined by basic trigonometry:

```
shadow_length = h_obj × ground_range / (h_sonar - h_obj)
```

Where `ground_range = √(slant_range² - h_sonar²)`

In [ ]:
def expected_shadow_length(h_obj, h_sonar, slant_range):
    """Compute expected acoustic shadow length from sonar geometry."""
    if h_sonar <= h_obj:
        return float('inf')  # object taller than sonar
    if h_obj <= 0 or h_sonar <= 0:
        return 0.0
    ground_range = math.sqrt(max(slant_range**2 - h_sonar**2, 0))
    shadow = (h_obj * ground_range) / (h_sonar - h_obj)
    return max(shadow, 0.0)

# Interactive demo
h_sonar = 10.0  # meters
print(f'Sonar altitude: {h_sonar}m\n')
print(f'{"Object Height":>15s} {"Slant Range":>12s} {"Shadow Length":>14s}')
print('-' * 45)
for h_obj in [0.2, 0.5, 1.0, 2.0, 3.0]:
    for sr in [15, 25, 40]:
        sl = expected_shadow_length(h_obj, h_sonar, sr)
        print(f'{h_obj:>13.1f}m {sr:>10d}m {sl:>12.2f}m')

In [ ]:
# ---- Geometry diagram ----
fig, ax = plt.subplots(figsize=(12, 6))

# Sonar, seabed, object, shadow
h_sonar, slant_r = 10, 25
h_obj = 2.0
ground_r = math.sqrt(slant_r**2 - h_sonar**2)
shadow_l = expected_shadow_length(h_obj, h_sonar, slant_r)

# Draw seabed
ax.axhline(y=0, color='saddlebrown', linewidth=3, label='Seabed')
ax.fill_between([0, ground_r + shadow_l + 5], -1, 0, color='saddlebrown', alpha=0.2)

# Sonar position
ax.plot(0, h_sonar, 'rv', markersize=15, label=f'Sonar (alt={h_sonar}m)')
ax.annotate('Sonar', xy=(0, h_sonar), xytext=(-3, h_sonar+0.5), fontsize=10)

# Object
ax.add_patch(plt.Rectangle((ground_r-0.3, 0), 0.6, h_obj, color='steelblue', label=f'Object (h={h_obj}m)'))

# Sonar beam to object top
ax.plot([0, ground_r], [h_sonar, h_obj], 'g--', alpha=0.5, label='Sonar beam')

# Shadow (behind object)
ax.fill_between([ground_r, ground_r + shadow_l], -0.3, 0,
                color='black', alpha=0.4, label=f'Shadow ({shadow_l:.1f}m)')

# Shadow projection line
ax.plot([0, ground_r + shadow_l], [h_sonar, 0], 'k:', alpha=0.3)

# Annotations
ax.annotate('', xy=(ground_r+shadow_l, -0.5), xytext=(ground_r, -0.5),
            arrowprops=dict(arrowstyle='<->', color='red'))
ax.text(ground_r + shadow_l/2, -0.8, f'{shadow_l:.1f}m', ha='center', color='red', fontsize=11)

ax.set_xlabel('Ground Range (m)', fontsize=12)
ax.set_ylabel('Height (m)', fontsize=12)
ax.set_title('Acoustic Shadow Geometry in Side-Scan Sonar', fontsize=14)
ax.legend(loc='upper right')
ax.set_ylim(-1.5, h_sonar + 2)
ax.set_xlim(-5, ground_r + shadow_l + 5)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Shadow Verification Logic

In [ ]:
class ShadowVerifier:
    def __init__(self, tolerance=0.4, min_penalty=0.1, max_penalty=0.5):
        self.tolerance = tolerance
        self.min_penalty = min_penalty
        self.max_penalty = max_penalty
    
    def verify(self, observed_shadow, expected_shadow):
        """Returns (is_consistent, penalty, ratio)."""
        if expected_shadow <= 0:
            return True, 0.0, 0.0
        ratio = observed_shadow / expected_shadow
        deviation = abs(ratio - 1.0)
        
        if deviation <= self.tolerance:
            return True, 0.0, ratio
        
        excess = deviation - self.tolerance
        penalty = min(self.min_penalty + excess * (self.max_penalty - self.min_penalty),
                      self.max_penalty)
        return False, penalty, ratio

verifier = ShadowVerifier()

# Test cases
print(f'{"Case":30s} {"Observed":>10s} {"Expected":>10s} {"Ratio":>7s} {"Consistent":>11s} {"Penalty":>8s}')
print('-' * 80)

cases = [
    ('Real debris (good shadow)', 4.5, 5.0),
    ('Real debris (slight noise)', 5.8, 5.0),
    ('Rock cluster (too short)', 1.2, 5.0),
    ('Rock cluster (no shadow)', 0.3, 5.0),
    ('Rock cluster (too long)', 12.0, 5.0),
    ('Flat object (no shadow)', 0.1, 0.5),
]

for name, obs, exp in cases:
    ok, penalty, ratio = verifier.verify(obs, exp)
    print(f'{name:30s} {obs:10.1f} {exp:10.1f} {ratio:7.2f} {"✓" if ok else "✗":>11s} {penalty:8.3f}')

## 3. Impact on Confidence Scores

In [ ]:
# Simulate detections with and without shadow filter
np.random.seed(42)

# Simulate 100 detections: 70 real debris, 30 rock-cluster FPs
n_real, n_rock = 70, 30

# Real debris: shadows are consistent (ratio ≈ 1.0 ± 0.2)
real_confs = np.random.uniform(0.4, 0.95, n_real)
real_shadow_ratios = np.random.normal(1.0, 0.15, n_real)

# Rock clusters: high confidence but inconsistent shadows
rock_confs = np.random.uniform(0.5, 0.9, n_rock)
rock_shadow_ratios = np.abs(np.random.normal(0.3, 0.3, n_rock))  # too short or wrong

all_confs = np.concatenate([real_confs, rock_confs])
all_ratios = np.concatenate([real_shadow_ratios, rock_shadow_ratios])
is_real = np.concatenate([np.ones(n_real), np.zeros(n_rock)])

# Apply shadow filter
filtered_confs = []
for conf, ratio in zip(all_confs, all_ratios):
    _, penalty, _ = verifier.verify(ratio, 1.0)  # 1.0 = expected ratio
    filtered_confs.append(max(conf - penalty, 0))
filtered_confs = np.array(filtered_confs)

# Visualise
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Before filter
ax1.scatter(all_confs[is_real==1], all_ratios[is_real==1], c='green', alpha=0.6, label='Real debris', s=50)
ax1.scatter(all_confs[is_real==0], all_ratios[is_real==0], c='red', alpha=0.6, label='Rock cluster', s=50)
ax1.axhline(y=1.0, color='blue', linestyle='--', alpha=0.3, label='Expected ratio')
ax1.axhspan(1-verifier.tolerance, 1+verifier.tolerance, alpha=0.1, color='blue', label='Tolerance band')
ax1.set_xlabel('Raw Confidence')
ax1.set_ylabel('Shadow Ratio (observed/expected)')
ax1.set_title('Before Shadow Filter')
ax1.legend()
ax1.grid(True, alpha=0.3)

# After filter
ax2.scatter(filtered_confs[is_real==1], all_ratios[is_real==1], c='green', alpha=0.6, label='Real debris', s=50)
ax2.scatter(filtered_confs[is_real==0], all_ratios[is_real==0], c='red', alpha=0.6, label='Rock cluster', s=50)
ax2.axvline(x=0.25, color='orange', linestyle='--', label='Conf threshold (0.25)')
ax2.set_xlabel('Filtered Confidence')
ax2.set_ylabel('Shadow Ratio')
ax2.set_title('After Shadow Filter')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Shadow Geometry Filter: Rock-Cluster FP Suppression', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Stats
thresh = 0.25
before_tp = ((all_confs >= thresh) & (is_real == 1)).sum()
before_fp = ((all_confs >= thresh) & (is_real == 0)).sum()
after_tp = ((filtered_confs >= thresh) & (is_real == 1)).sum()
after_fp = ((filtered_confs >= thresh) & (is_real == 0)).sum()

print(f'\nAt conf threshold {thresh}:')
print(f'  Before filter: TP={before_tp}, FP={before_fp}, Precision={before_tp/(before_tp+before_fp):.3f}')
print(f'  After filter:  TP={after_tp},  FP={after_fp},  Precision={after_tp/(after_tp+after_fp+1e-10):.3f}')
print(f'  FP reduction: {(1-after_fp/max(before_fp,1))*100:.0f}%')

## 4. Shadow Extraction from Sonar Image

In [ ]:
def extract_shadow(image, bbox, search_factor=2.0, thresh_ratio=0.4):
    """Extract acoustic shadow region below a detection."""
    h, w = image.shape[:2]
    x1, y1, x2, y2 = [int(v) for v in bbox]
    obj_h = y2 - y1
    
    # Search below object
    sy_start = y2
    sy_end = min(y2 + int(obj_h * search_factor), h)
    sx_start = max(x1 - 5, 0)
    sx_end = min(x2 + 5, w)
    
    if sy_end <= sy_start: return None, 0
    
    roi = image[sy_start:sy_end, sx_start:sx_end]
    if roi.size == 0: return None, 0
    
    shadow_thresh = max(roi.mean() * thresh_ratio, 10)
    shadow_mask = (roi < shadow_thresh).astype(np.uint8)
    
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    shadow_mask = cv2.morphologyEx(shadow_mask, cv2.MORPH_OPEN, kernel)
    
    cols = shadow_mask.any(axis=1)
    if cols.any():
        rows = np.where(cols)[0]
        length = rows[-1] - rows[0] + 1
    else:
        length = 0
    
    return shadow_mask, length

# Demo with synthetic image
from numpy.random import RandomState
rng = RandomState(42)

# Create a fake sonar image with an object and shadow
demo_img = np.full((200, 300), 100, dtype=np.uint8)
demo_img += rng.randint(-15, 15, demo_img.shape).astype(np.uint8)

# Bright object
demo_img[60:90, 120:180] = 220
# Dark shadow below
demo_img[90:130, 120:180] = 15

bbox = [120, 60, 180, 90]
shadow_mask, shadow_len = extract_shadow(demo_img, bbox)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.imshow(demo_img, cmap='gray')
rect = plt.Rectangle((bbox[0], bbox[1]), bbox[2]-bbox[0], bbox[3]-bbox[1],
                       linewidth=2, edgecolor='lime', facecolor='none')
ax1.add_patch(rect)
ax1.set_title(f'Sonar Image (object in green box)')
ax1.axis('off')

if shadow_mask is not None:
    ax2.imshow(shadow_mask, cmap='gray')
    ax2.set_title(f'Extracted Shadow (length={shadow_len}px)')
else:
    ax2.text(0.5, 0.5, 'No shadow detected', transform=ax2.transAxes, ha='center')
ax2.axis('off')

plt.tight_layout()
plt.show()

## 5. Summary

The shadow geometry filter is a **deterministic, zero-overhead** post-NMS check:
- No second trained network needed
- Pure trigonometry: compare predicted vs. observed shadow length
- Rock clusters (strong return, wrong shadow) get penalised
- Real raised debris (consistent shadow) passes through

Implemented in `ml/inference/shadow_verification.py`, called by
`ml/inference/confidence_filter.py` after NMS + Platt scaling.

**Next:** `07_export_and_benchmark.ipynb`